# LeetCode 558 — Logical OR of Two Binary Grids Represented as Quad-Trees bằng Python

Link đề: https://leetcode.com/problems/logical-or-of-two-binary-grids-represented-as-quad-trees/ · Độ khó: **Medium** · Chủ đề: Tree, Divide and Conquer

Skeleton LeetCode:

```python
"""
# Definition for a QuadTree node.
class Node:
    def __init__(self, val, isLeaf, topLeft, topRight, bottomLeft, bottomRight):
        self.val = val
        self.isLeaf = isLeaf
        self.topLeft = topLeft
        self.topRight = topRight
        self.bottomLeft = bottomLeft
        self.bottomRight = bottomRight
"""

class Solution:
    def intersect(self, quadTree1: 'Node', quadTree2: 'Node') -> 'Node':
        ...
```

Ghi chú: trên LeetCode, lớp `Node` được cho sẵn (phần chú thích). Trong notebook này ta **tự dựng lại** `Node`, hàm dựng cây từ lưới, và hàm tuần tự hóa (serialize) theo kiểu LeetCode để có thể chạy và kiểm tra mọi thứ ngay tại đây. Chữ ký của `Node` trong đề gốc có thể khác chút giữa các thời điểm (có/không giá trị mặc định cho đối số); vì vậy code nộp của ta **luôn truyền đủ 6 đối số** khi tạo `Node`. Phần mô tả quy ước serialize dưới đây viết theo hiểu biết về đề; nếu muốn chắc chắn hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu **quad-tree** (cây tứ phân) biểu diễn một lưới nhị phân `n x n` (`n` là lũy thừa của 2): nút lá = một khối đồng nhất toàn `0` hoặc toàn `1`; nút trong = khối hỗn hợp, chia thành 4 góc phần tư.
2. Nắm các công cụ Python cần thiết: định nghĩa lớp `Node`, đệ quy chia 4, hàng đợi `collections.deque` để duyệt theo tầng (BFS), `all(...)` với generator, toán tử `or` trên giá trị `bool`, và so sánh đối tượng bằng `is`.
3. Đi từ **bung cả hai cây ra lưới rồi OR từng ô và dựng lại cây** → **đệ quy trực tiếp trên hai cây (cách nộp)** → **trộn bằng cách "bung ảo" lá rồi nén ở pass thứ hai** → **bản lặp dùng ngăn xếp tường minh**.
4. Hiểu mấu chốt: gặp **lá `1`** thì kết quả là lá `1` (1 OR gì cũng là 1); gặp **lá `0`** thì kết quả chính là cây còn lại; chỉ khi **cả hai đều là nút trong** mới phải đệ quy 4 góc, và sau đó **nén lại** nếu 4 con đều là lá cùng giá trị.
5. Tránh các lỗi kinh điển: quên nén 4 lá cùng giá trị, nhầm `val` của nút **trong** (giá trị tùy ý) là có nghĩa, trả nhầm lá `0` thay vì cây còn lại, và kiểm tra "cùng giá trị" mà quên kiểm tra các con có phải lá không.

## 2. Đề bài & ví dụ

Một **ma trận nhị phân** chỉ chứa `0` và `1`. Cho `quadTree1` và `quadTree2`, mỗi cây biểu diễn một ma trận nhị phân `n x n`. Hãy trả về **quad-tree** biểu diễn ma trận là kết quả phép **OR từng bit** (logical bitwise OR) của hai ma trận đó.

**Quad-tree** là cây mà mỗi nút trong có đúng 4 con (`topLeft`, `topRight`, `bottomLeft`, `bottomRight`), mỗi nút có hai thuộc tính:

- `val`: `True` nếu nút biểu diễn khối toàn `1`, `False` nếu toàn `0`.
- `isLeaf`: `True` nếu nút là lá, `False` nếu có 4 con.

**Cách dựng cây từ một vùng của lưới:**

1. Nếu vùng đang xét toàn cùng một giá trị: đặt `isLeaf = True`, `val` = giá trị đó, 4 con là `null`, dừng.
2. Ngược lại: đặt `isLeaf = False`, `val` = **giá trị bất kỳ** (cả `True` lẫn `False` đều được chấp nhận), chia vùng thành 4 phần bằng nhau, và đệ quy cho từng con.

**Định dạng serialize (input/output của LeetCode):** duyệt theo tầng (level order); mỗi nút là cặp `[isLeaf, val]` với `True -> 1`, `False -> 0`; `null` đánh dấu chỗ không có nút (giống tuần tự hóa cây nhị phân). Các `null` ở cuối được bỏ.

**Ví dụ 1** (lưới `4 x 4`)

- Input: `quadTree1 = [[0,1],[1,1],[1,1],[1,0],[1,0]]`, `quadTree2 = [[0,1],[1,1],[0,1],[1,1],[1,0],null,null,null,null,[1,0],[1,0],[1,1],[1,1]]`
- Output: `[[0,0],[1,1],[1,1],[1,1],[1,0]]`

**Ví dụ 2** (lưới `1 x 1`)

- Input: `quadTree1 = [[1,0]]`, `quadTree2 = [[1,0]]`
- Output: `[[1,0]]` (hai ma trận `1x1` chứa `0`, kết quả cũng `0`)

**Ràng buộc:**

- `quadTree1` và `quadTree2` đều là quad-tree **hợp lệ**, mỗi cây biểu diễn một lưới `n x n`.
- `n == 2^x` với `0 <= x <= 9` (tức `n` tối đa `512`).

Lưu ý quan trọng: ta **không được phép** (và không cần) dựng lưới `n x n` để giải bài này (ví dụ trong đề ghi rõ). Tuy vậy, ở phần dưới ta vẫn dựng lưới để **kiểm thử** và làm cách baseline.

## 3. Kiến thức nền

Vì LeetCode cho sẵn lớp `Node` và thực hiện serialize/deserialize ở phía hệ thống, ta tự dựng chúng ở đây. Hãy đọc từng mục và chạy cell tương ứng.

### 3.1. Lớp `Node` cho quad-tree

Mỗi nút có 6 trường: `val`, `isLeaf`, và 4 con. Với nút lá, 4 con là `None`.

In [1]:
from typing import Optional, List
from collections import deque
import random
import time


class Node:
    def __init__(self, val=False, isLeaf=False, topLeft=None, topRight=None, bottomLeft=None, bottomRight=None):
        self.val = val
        self.isLeaf = isLeaf
        self.topLeft = topLeft
        self.topRight = topRight
        self.bottomLeft = bottomLeft
        self.bottomRight = bottomRight

    def children(self):
        return [self.topLeft, self.topRight, self.bottomLeft, self.bottomRight]


def leaf(v):
    return Node(bool(v), True, None, None, None, None)

# Ví dụ nhỏ: lưới 2x2 [[1, 0], [0, 1]] -> nút gốc không phải lá, 4 con là 4 lá
root = Node(False, False, leaf(1), leaf(0), leaf(0), leaf(1))
print("gốc isLeaf =", root.isLeaf, "| 4 con có val:", [c.val for c in root.children()])

gốc isLeaf = False | 4 con có val: [True, False, False, True]


### 3.2. Dựng cây từ lưới và ngược lại: `build`, `to_grid`

- `build(grid)`: nếu cả vùng đồng nhất thì tạo lá; ngược lại chia 4 và đệ quy. Ta dùng ba tham số `(r, c, size)` để chỉ vùng vuông có góc trên trái `(r, c)` và cạnh `size` — không cần cắt lưới, nên không tốn bộ nhớ sao chép.
- `to_grid(node, n)`: bung cây thành lưới `n x n` (để kiểm tra).

Khi nút trong, ta đặt `val = False` (đề cho phép giá trị bất kỳ). Hàm `all(... for ...)` dùng generator để kiểm tra "mọi ô đều bằng ô đầu tiên".

In [2]:
def build(grid):
    n = len(grid)

    def rec(r, c, size):
        first = grid[r][c]
        if all(grid[y][x] == first for y in range(r, r + size) for x in range(c, c + size)):
            return leaf(first)
        h = size // 2
        return Node(False, False,
                    rec(r, c, h), rec(r, c + h, h),
                    rec(r + h, c, h), rec(r + h, c + h, h))

    return rec(0, 0, n)


def to_grid(node, n):
    g = [[0] * n for _ in range(n)]

    def fill(nd, r, c, size):
        if nd.isLeaf:
            if nd.val:
                for y in range(r, r + size):
                    for x in range(c, c + size):
                        g[y][x] = 1
            return
        h = size // 2
        fill(nd.topLeft, r, c, h)
        fill(nd.topRight, r, c + h, h)
        fill(nd.bottomLeft, r + h, c, h)
        fill(nd.bottomRight, r + h, c + h, h)

    fill(node, 0, 0, n)
    return g


grid4 = [
    [1, 1, 0, 0],
    [1, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1],
]
t = build(grid4)
print("gốc là lá?", t.isLeaf)
print("góc trên trái là lá", t.topLeft.isLeaf, "val", t.topLeft.val)
print("góc dưới phải là lá?", t.bottomRight.isLeaf, "(khối 2x2 [[1,0],[0,1]] hỗn hợp)")
print(to_grid(t, 4) == grid4)

gốc là lá? False
góc trên trái là lá True val True
góc dưới phải là lá? False (khối 2x2 [[1,0],[0,1]] hỗn hợp)
True


### 3.3. In cây ra cho dễ nhìn: `show`

Một hàm in theo tiền thứ tự (pre-order), thụt lề theo độ sâu, giúp bạn "nhìn thấy" hình dạng cây.

In [3]:
def show(node, indent=0, name="gốc"):
    pad = "  " * indent
    if node.isLeaf:
        print(f"{pad}{name}: lá {int(node.val)}")
    else:
        print(f"{pad}{name}: nút trong (val={int(node.val)})")
        for nm, ch in zip(["TL", "TR", "BL", "BR"], node.children()):
            show(ch, indent + 1, nm)

show(t)

gốc: nút trong (val=0)
  TL: lá 1
  TR: lá 0
  BL: lá 0
  BR: nút trong (val=0)
    TL: lá 1
    TR: lá 0
    BL: lá 0
    BR: lá 1


### 3.4. Serialize kiểu LeetCode bằng hàng đợi `deque`

Quy tắc: duyệt theo tầng (BFS). Dùng `collections.deque` làm hàng đợi (`popleft()` lấy phần tử đầu, `append`/`extend` thêm vào cuối, đều `O(1)`).

- Lấy một nút ra: nếu là `None` thì ghi `null`; ngược lại ghi `[isLeaf, val]` (chuyển `True/False` thành `1/0`).
- **Mọi nút khác `None`** đều đẩy **4 con** của nó vào hàng đợi; với nút **lá**, 4 con đó là `None` (sẽ được ghi thành 4 `null` ở tầng sau).
- Cuối cùng bỏ các `null` thừa ở cuối.

Deserialize làm ngược lại: dựng gốc từ phần tử đầu; mỗi lần lấy một nút khác `None` ra khỏi hàng đợi, đọc tiếp **4 phần tử** làm 4 con (nếu là nút lá thì 4 phần tử này là `null`).

In [4]:
def serialize(root):
    if root is None:
        return []
    out = []
    q = deque([root])
    while q:
        nd = q.popleft()
        if nd is None:
            out.append(None)
            continue
        out.append([int(nd.isLeaf), int(nd.val)])
        if nd.isLeaf:
            q.extend([None, None, None, None])
        else:
            q.extend(nd.children())
    while out and out[-1] is None:
        out.pop()
    return out


def deserialize(data):
    if not data:
        return None

    def mk(item):
        return None if item is None else Node(bool(item[1]), bool(item[0]), None, None, None, None)

    root = mk(data[0])
    q = deque([root])
    idx = 1
    while q:
        nd = q.popleft()
        kids = []
        for _ in range(4):
            kids.append(mk(data[idx]) if idx < len(data) else None)
            idx += 1
        if nd.isLeaf:
            continue                      # lá: 4 phần tử vừa đọc đều là null
        nd.topLeft, nd.topRight, nd.bottomLeft, nd.bottomRight = kids
        q.extend(kids)
    return root


print(serialize(t))
ex2_tree = [[0, 1], [1, 1], [0, 1], [1, 1], [1, 0], None, None, None, None, [1, 0], [1, 0], [1, 1], [1, 1]]
print(serialize(deserialize(ex2_tree)) == ex2_tree)         # vòng khứ hồi phải khớp ví dụ trong đề
print(serialize(deserialize([[1, 0]])))

[[0, 0], [1, 1], [1, 0], [1, 0], [0, 0], None, None, None, None, None, None, None, None, None, None, None, None, [1, 1], [1, 0], [1, 0], [1, 1]]
True
[[1, 0]]


Quan sát: ví dụ `ex2_tree` (cây thứ hai của ví dụ 1 trong đề) được đọc rồi ghi lại **đúng nguyên văn**, chứng tỏ hàm `serialize`/`deserialize` của ta đúng quy ước của đề. Bốn `null` liên tiếp là bốn con của lá `[1,1]` đầu tiên.

### 3.5. `or` trên giá trị `bool` và ý nghĩa OR từng ô

Với `bool`, `a or b` cho `True` khi ít nhất một vế là `True`. Phép OR từng bit của hai ô: `0|0 = 0`, còn lại bằng `1`. Hai điều rút ra **rất quan trọng cho bài này**:

- `1 OR x = 1` với mọi `x` (không cần biết `x`).
- `0 OR x = x` (kết quả giống hệt `x`).

In [5]:
for a in (False, True):
    for b in (False, True):
        print(f"{a!s:5} or {b!s:5} = {a or b!s:5} | {int(a)} | {int(b)} = {int(a) | int(b)}")

False or False = False | 0 | 0 = 0
False or True  = True  | 0 | 1 = 1
True  or False = True  | 1 | 0 = 1
True  or True  = True  | 1 | 1 = 1


### 3.6. Chia 4 góc bằng đệ quy và dùng chung nút (`is`)

Trong bài này ta sẽ **trả về nguyên một nút của cây đầu vào** (không sao chép) khi kết quả đúng bằng cây con đó. Hai biến cùng trỏ một đối tượng được kiểm bằng `is`. Điều này an toàn nếu ta **không sửa** các nút sau khi trả về, và rất tiết kiệm bộ nhớ/thời gian.

In [6]:
a = leaf(1)
b = a                  # cùng một đối tượng
c = leaf(1)            # đối tượng khác nhưng cùng nội dung
print(a is b, a is c, a.val == c.val)

True False True


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Hãy làm tay với **ví dụ 1** (lưới `4 x 4`). Đọc cây từ mục 2:

- `quadTree1`: gốc là nút trong, 4 con là lá `1, 1, 0, 0` (nửa trên toàn `1`, nửa dưới toàn `0`).
- `quadTree2`: góc trên trái là lá `1`; góc trên phải là nút trong mà bốn con (theo thứ tự TL, TR, BL, BR) là lá `0, 0, 1, 1`; góc dưới trái là lá `1`; góc dưới phải là lá `0`.

OR từng góc phần tư:

| Góc | `quadTree1` | `quadTree2` | Kết quả | Lý do |
|---|---|---|---|---|
| TL | lá `1` | lá `1` | lá `1` | `1 OR 1` |
| TR | lá `1` | nút trong | lá `1` | `1 OR bất kỳ = 1`, **không cần nhìn** cây thứ hai |
| BL | lá `0` | lá `1` | lá `1` | `0 OR x = x` |
| BR | lá `0` | lá `0` | lá `0` | `0 OR 0` |

Bốn con kết quả: `1, 1, 1, 0`, không đồng nhất nên giữ gốc là nút trong. Đúng với đáp án `[[0,0],[1,1],[1,1],[1,1],[1,0]]`.

Trực giác rút ra:

1. Nếu **một bên là lá**: kết quả hoặc là lá `1` (khi lá đó là `1`) hoặc là **cây của bên kia** (khi lá đó là `0`). **Không phải đệ quy.**
2. Chỉ khi **cả hai bên là nút trong** mới phải đệ quy 4 góc.
3. Sau khi có 4 con kết quả, nếu cả 4 đều là **lá cùng giá trị** thì khối thực ra đồng nhất, theo định nghĩa phải là **một lá** — ta phải **nén** lại. (Ví dụ `[[1,0],[0,1]]` OR `[[0,1],[1,0]]` toàn `1`.)

In [7]:
t1 = deserialize([[0, 1], [1, 1], [1, 1], [1, 0], [1, 0]])
t2 = deserialize(ex2_tree)
g1, g2 = to_grid(t1, 4), to_grid(t2, 4)
print("lưới 1:"); [print("  ", row) for row in g1]
print("lưới 2:"); [print("  ", row) for row in g2]
g_or = [[a | b for a, b in zip(r1, r2)] for r1, r2 in zip(g1, g2)]
print("OR từng ô:"); [print("  ", row) for row in g_or]
print("cây của lưới OR:", serialize(build(g_or)))

lưới 1:
   [1, 1, 1, 1]
   [1, 1, 1, 1]
   [0, 0, 0, 0]
   [0, 0, 0, 0]
lưới 2:
   [1, 1, 0, 0]
   [1, 1, 1, 1]
   [1, 1, 0, 0]
   [1, 1, 0, 0]
OR từng ô:
   [1, 1, 1, 1]
   [1, 1, 1, 1]
   [1, 1, 0, 0]
   [1, 1, 0, 0]
cây của lưới OR: [[0, 0], [1, 1], [1, 1], [1, 1], [1, 0]]


## 5. Cách 1 — Brute Force: bung ra lưới, OR từng ô, dựng lại cây

### Ý tưởng

Đúng như đề cảnh báo: cách "thật thà" nhất là
1. Bung hai cây thành hai lưới `n x n`.
2. OR từng ô.
3. Dựng lại quad-tree từ lưới kết quả bằng `build`.

Chỉ có một điểm tinh tế: đề **không cho biết `n`**. Nhưng cây cho ta biết độ cao `h` (số tầng nút trong tối đa), và lưới cần ít nhất `n = 2^h`. Ta lấy `h = max(độ cao 2 cây)` và `n = 2^h` — cây kết quả không phụ thuộc vào việc ta chọn `n` lớn hơn hay không, vì mọi lá chỉ là "khối đồng nhất".

### Code

In [8]:
def height(node):
    if node.isLeaf:
        return 0
    return 1 + max(height(c) for c in node.children())


def intersect_grid(t1, t2):
    n = 2 ** max(height(t1), height(t2))
    g1, g2 = to_grid(t1, n), to_grid(t2, n)
    g = [[a | b for a, b in zip(r1, r2)] for r1, r2 in zip(g1, g2)]
    return build(g)


r = intersect_grid(t1, t2)
print(serialize(r))
print(serialize(r) == [[0, 0], [1, 1], [1, 1], [1, 1], [1, 0]])
print(serialize(intersect_grid(deserialize([[1, 0]]), deserialize([[1, 0]]))))

[[0, 0], [1, 1], [1, 1], [1, 1], [1, 0]]
True
[[1, 0]]


### Độ phức tạp và nút thắt

- Time: `O(n^2)` để bung mỗi cây, OR, rồi `build` (nén lặp lại `all(...)` trên mỗi tầng nên thực tế `O(n^2 log n)`). Với `n = 512`: hơn `260 000` ô.
- Space: `O(n^2)` cho các lưới.

**Nút thắt:** ta bung cả những khối khổng lồ toàn `0` hoặc toàn `1` thành từng ô, trong khi cây đã **nén sẵn** thông tin đó. Ví dụ nếu `quadTree1` chỉ là một lá `1` (lưới toàn `1`) thì đáp án hiển nhiên là lá `1` mà ta vẫn tốn `O(n^2)`. Hãy xem chi phí tăng thế nào: một cây chỉ gồm một lá nhưng cây kia sâu.

In [9]:
def deep_tree(depth):
    # cây "đường chéo": mỗi tầng chỉ góc trên trái tiếp tục chia, ba góc còn lại là lá 0, góc cuối cùng là 1 ô = 1
    if depth == 0:
        return leaf(1)
    return Node(False, False, deep_tree(depth - 1), leaf(0), leaf(0), leaf(0))

for depth in [3, 6, 9]:
    big, one = deep_tree(depth), leaf(1)
    t0 = time.perf_counter()
    res = intersect_grid(big, one)
    print(f"n={2 ** depth}: cách 1 mất {time.perf_counter() - t0:.4f}s | kết quả serialize = {serialize(res)}")

n=8: cách 1 mất 0.0000s | kết quả serialize = [[1, 1]]
n=64: cách 1 mất 0.0004s | kết quả serialize = [[1, 1]]
n=512: cách 1 mất 0.0362s | kết quả serialize = [[1, 1]]


Dù đáp án luôn là **lá `1`** (vì `quadTree2` là lá `1`), thời gian tăng gấp ~4 lần mỗi khi `n` gấp đôi (tỉ lệ `n^2`). Ta muốn một thuật toán chỉ xem xét **cấu trúc cây**, không xem từng ô.

## 6. Ý tưởng thuật toán / bước đột phá

Bước đột phá là **làm việc trực tiếp trên hai cây**, đi xuống **đồng thời** theo cùng một vị trí. Gọi `dfs(a, b)` trả về quad-tree (đã chuẩn) của OR hai khối mà `a` và `b` biểu diễn. Có 3 trường hợp:

1. **`a` là lá:**
   - `a.val = 1`: toàn `1`, OR với gì cũng toàn `1`. Trả `a`.
   - `a.val = 0`: toàn `0`, OR giữ nguyên `b`. Trả `b`.
2. **`b` là lá:** đối xứng, `b.val` thì trả `b` còn không trả `a`.
3. **Cả hai là nút trong:** đệ quy cho 4 góc. Gọi 4 kết quả là `tl, tr, bl, br`. Nếu cả 4 đều là **lá** và **cùng `val`**, khối thật ra đồng nhất: trả **một lá** với `val` đó. Ngược lại tạo nút trong mới với 4 con này.

Có hai điểm cần giải thích:

- Trường hợp 1 và 2 gộp luôn cả trường hợp "hai lá" (nếu `a` là lá `0` ta trả `b`, mà `b` cũng là lá nên đó đúng là `0 OR b`).
- Vì sao chỉ cần nén **một cấp**? Các con `tl, tr, bl, br` mà ta vừa nhận từ đệ quy đã là cây **chuẩn** (đúng bằng quy nạp). Nếu một con là **nút trong** thì khối của nó hỗn hợp nên khối cha cũng hỗn hợp, không thể nén. Nếu cả 4 đều là lá, ta so `val`; chỉ khi bằng nhau mới nén. Nén một cấp là đủ, và sau khi nén ta trả về lá để cấp cha tự xét tiếp.

Cuối cùng, khi trả nút trong mới, ta đặt `val = False` (đề cho phép tùy ý).

In [10]:
def dfs_demo(a, b, depth=0):
    pad = "  " * depth
    if a.isLeaf:
        print(f"{pad}a là lá {int(a.val)} ->", "trả a" if a.val else "trả b")
        return a if a.val else b
    if b.isLeaf:
        print(f"{pad}b là lá {int(b.val)} ->", "trả b" if b.val else "trả a")
        return b if b.val else a
    print(f"{pad}cả hai là nút trong -> đệ quy 4 góc")
    tl = dfs_demo(a.topLeft, b.topLeft, depth + 1)
    tr = dfs_demo(a.topRight, b.topRight, depth + 1)
    bl = dfs_demo(a.bottomLeft, b.bottomLeft, depth + 1)
    br = dfs_demo(a.bottomRight, b.bottomRight, depth + 1)
    if tl.isLeaf and tr.isLeaf and bl.isLeaf and br.isLeaf and tl.val == tr.val == bl.val == br.val:
        print(f"{pad}4 con đều là lá {int(tl.val)} -> nén thành 1 lá")
        return leaf(tl.val)
    return Node(False, False, tl, tr, bl, br)

res = dfs_demo(t1, t2)
print(serialize(res))

cả hai là nút trong -> đệ quy 4 góc
  a là lá 1 -> trả a
  a là lá 1 -> trả a
  a là lá 0 -> trả b
  a là lá 0 -> trả b
[[0, 0], [1, 1], [1, 1], [1, 1], [1, 0]]


## 7. Các cách tiếp theo

Ta có 4 cách khác nhau thực sự:

| Cách | Ý tưởng |
|---|---|
| 1 | Bung ra lưới, OR từng ô, dựng lại cây (mục 5) |
| 2 | Đệ quy trực tiếp trên hai cây, lá `1` thống trị, lá `0` nhường (**cách nộp**) |
| 3 | **Bung ảo** lá thành 4 con giống nó để trộn đồng nhất, rồi **nén ở pass thứ hai** |
| 4 | Bản **lặp**: ngăn xếp tường minh (hậu thứ tự), không đệ quy |

### Cách 2 — Đệ quy trực tiếp trên hai cây

**Giải thích từng đoạn:**

- `if t1.isLeaf: return t1 if t1.val else t2`: `t1` là lá `1` thì kết quả là `t1` (toàn `1`); là lá `0` thì kết quả là `t2`.
- `if t2.isLeaf: return t2 if t2.val else t1`: đối xứng.
- Tới đây cả hai là nút trong. Đệ quy bốn góc theo cùng vị trí.
- Nếu 4 con đều là lá và cùng `val`: trả một lá mới. Ngược lại trả nút trong mới `Node(False, False, tl, tr, bl, br)`.

### Code demo

In [11]:
def intersect_dfs(t1, t2):
    if t1.isLeaf:
        return t1 if t1.val else t2
    if t2.isLeaf:
        return t2 if t2.val else t1
    tl = intersect_dfs(t1.topLeft, t2.topLeft)
    tr = intersect_dfs(t1.topRight, t2.topRight)
    bl = intersect_dfs(t1.bottomLeft, t2.bottomLeft)
    br = intersect_dfs(t1.bottomRight, t2.bottomRight)
    if tl.isLeaf and tr.isLeaf and bl.isLeaf and br.isLeaf and tl.val == tr.val == bl.val == br.val:
        return Node(tl.val, True, None, None, None, None)
    return Node(False, False, tl, tr, bl, br)

print(serialize(intersect_dfs(t1, t2)))
print(serialize(intersect_dfs(deserialize([[1, 0]]), deserialize([[1, 0]]))))

[[0, 0], [1, 1], [1, 1], [1, 1], [1, 0]]
[[1, 0]]


### Code nộp LeetCode (Cách 2)

```python
class Solution:
    def intersect(self, quadTree1: 'Node', quadTree2: 'Node') -> 'Node':
        def dfs(t1, t2):
            if t1.isLeaf:
                return t1 if t1.val else t2
            if t2.isLeaf:
                return t2 if t2.val else t1
            tl = dfs(t1.topLeft, t2.topLeft)
            tr = dfs(t1.topRight, t2.topRight)
            bl = dfs(t1.bottomLeft, t2.bottomLeft)
            br = dfs(t1.bottomRight, t2.bottomRight)
            if tl.isLeaf and tr.isLeaf and bl.isLeaf and br.isLeaf and tl.val == tr.val == bl.val == br.val:
                return Node(tl.val, True, None, None, None, None)
            return Node(False, False, tl, tr, bl, br)

        return dfs(quadTree1, quadTree2)
```

### Độ phức tạp

- Time: `O(min(N1, N2))` với `N1`, `N2` là số nút của hai cây: ta chỉ đệ quy tại những vị trí mà **cả hai cây đều có nút trong**. Tệ nhất là `O(n^2)` khi hai cây đều sâu đến từng ô.
- Space: `O(log n)` ngăn xếp đệ quy (độ sâu tối đa `x + 1 <= 10`), chưa tính cây kết quả mới tạo.

### Cách 3 — "Bung ảo" lá rồi nén ở pass thứ hai

**Ý tưởng khác biệt:** thay vì phân tích trường hợp (lá `1`, lá `0`), ta coi **một lá `L` đứng cạnh một nút trong** như thể `L` cũng có 4 con, mỗi con là chính `L` ("bung ảo" lá thành 4 con giống nhau). Khi đó chỉ còn hai trường hợp:

- Cả hai là lá: trả lá `a.val or b.val`.
- Ngược lại: lấy 4 cặp con tương ứng (lá thì dùng chính nó cho cả 4), đệ quy, rồi tạo nút trong.

Kết quả của pass 1 **chưa chuẩn** (có thể có nút trong mà 4 con là lá cùng giá trị). Pass 2 `normalize` đi hậu thứ tự để nén. Việc tách "trộn" và "nén" làm mỗi hàm đơn giản hơn, nhưng đổi lại có thể duyệt nhiều nút hơn (ví dụ lá `1` đứng cạnh một cây sâu vẫn bị bung theo cấu trúc cây sâu).

### Code demo

In [12]:
def kids_of(node):
    # lá được "bung ảo" thành 4 con là chính nó
    return (node, node, node, node) if node.isLeaf else tuple(node.children())


def merge_virtual(a, b):
    if a.isLeaf and b.isLeaf:
        return leaf(a.val or b.val)
    parts = [merge_virtual(x, y) for x, y in zip(kids_of(a), kids_of(b))]
    return Node(False, False, *parts)


def normalize(node):
    if node.isLeaf:
        return node
    parts = [normalize(c) for c in node.children()]
    if all(p.isLeaf for p in parts) and len({p.val for p in parts}) == 1:
        return leaf(parts[0].val)
    return Node(False, False, *parts)


def intersect_two_pass(t1, t2):
    return normalize(merge_virtual(t1, t2))

raw = merge_virtual(t1, t2)
print("pass 1 (chưa nén):", serialize(raw))
print("pass 2 (đã nén)  :", serialize(normalize(raw)))

pass 1 (chưa nén): [[0, 0], [1, 1], [0, 0], [1, 1], [1, 0], None, None, None, None, [1, 1], [1, 1], [1, 1], [1, 1]]
pass 2 (đã nén)  : [[0, 0], [1, 1], [1, 1], [1, 1], [1, 0]]


Ở ví dụ 1, pass 1 để lại góc trên phải là một **nút trong có 4 lá `1`** (vì lá `1` của cây thứ nhất bị "bung ảo" theo cấu trúc của cây thứ hai); pass 2 nén nó thành một lá `1`, ra đúng `[[0,0],[1,1],[1,1],[1,1],[1,0]]`. Bên dưới là ví dụ pass 1 để lại nút thừa ở cả gốc: hai lưới `2 x 2` bổ sung nhau.

In [13]:
a = build([[1, 0], [0, 1]])
b = build([[0, 1], [1, 0]])
raw = merge_virtual(a, b)
print("trước nén:", serialize(raw))                       # gốc nút trong, 4 con lá 1
print("sau nén  :", serialize(normalize(raw)))            # chỉ còn một lá 1

trước nén: [[0, 0], [1, 1], [1, 1], [1, 1], [1, 1]]
sau nén  : [[1, 1]]


### Code nộp LeetCode (Cách 3)

```python
class Solution:
    def intersect(self, quadTree1: 'Node', quadTree2: 'Node') -> 'Node':
        def kids(node):
            return (node,) * 4 if node.isLeaf else (
                node.topLeft, node.topRight, node.bottomLeft, node.bottomRight)

        def merge(a, b):
            if a.isLeaf and b.isLeaf:
                return Node(bool(a.val or b.val), True, None, None, None, None)
            parts = [merge(x, y) for x, y in zip(kids(a), kids(b))]
            return Node(False, False, *parts)

        def normalize(node):
            if node.isLeaf:
                return node
            parts = [normalize(c) for c in (
                node.topLeft, node.topRight, node.bottomLeft, node.bottomRight)]
            if all(p.isLeaf for p in parts) and len({p.val for p in parts}) == 1:
                return Node(parts[0].val, True, None, None, None, None)
            return Node(False, False, *parts)

        return normalize(merge(quadTree1, quadTree2))
```

### Độ phức tạp

- Time: `O(N1 + N2)` trong trường hợp xấu (mỗi lá có thể bị bung theo cấu trúc cây còn lại); tệ nhất `O(n^2)`.
- Space: `O(log n)` ngăn xếp cộng cây trung gian `O(N)`.

### Cách 4 — Bản lặp với ngăn xếp tường minh

**Ý tưởng:** đệ quy hậu thứ tự có thể viết bằng **hai ngăn xếp**:

- `work`: các việc cần làm, mỗi việc là `(a, b, expanded)`.
- `res`: các kết quả đã tính xong.

Lấy một việc `(a, b, False)` ra: nếu là trường hợp dừng (có lá) thì đẩy kết quả vào `res`. Nếu cả hai là nút trong thì đẩy lại chính nó với `expanded = True` (để xử lý **sau** khi 4 con xong), rồi đẩy 4 việc con theo thứ tự **ngược** (BR, BL, TR, TL) để `TL` được xử lý trước. Khi lấy ra một việc `expanded = True`, bốn kết quả của con đang nằm trên đỉnh `res` (theo thứ tự BR ở trên cùng), ta lấy ra rồi ghép/nén.

### Code demo

In [14]:
def intersect_iter(t1, t2):
    work = [(t1, t2, False)]
    res = []
    while work:
        a, b, expanded = work.pop()
        if not expanded:
            if a.isLeaf:
                res.append(a if a.val else b)
                continue
            if b.isLeaf:
                res.append(b if b.val else a)
                continue
            work.append((a, b, True))
            # đẩy ngược để TL được lấy ra trước
            work.append((a.bottomRight, b.bottomRight, False))
            work.append((a.bottomLeft, b.bottomLeft, False))
            work.append((a.topRight, b.topRight, False))
            work.append((a.topLeft, b.topLeft, False))
        else:
            br = res.pop()
            bl = res.pop()
            tr = res.pop()
            tl = res.pop()
            if tl.isLeaf and tr.isLeaf and bl.isLeaf and br.isLeaf and tl.val == tr.val == bl.val == br.val:
                res.append(Node(tl.val, True, None, None, None, None))
            else:
                res.append(Node(False, False, tl, tr, bl, br))
    return res[0]

print(serialize(intersect_iter(t1, t2)))
print(serialize(intersect_iter(a, b)))

[[0, 0], [1, 1], [1, 1], [1, 1], [1, 0]]
[[1, 1]]


### Code nộp LeetCode (Cách 4)

```python
class Solution:
    def intersect(self, quadTree1: 'Node', quadTree2: 'Node') -> 'Node':
        work = [(quadTree1, quadTree2, False)]
        res = []
        while work:
            a, b, expanded = work.pop()
            if not expanded:
                if a.isLeaf:
                    res.append(a if a.val else b)
                    continue
                if b.isLeaf:
                    res.append(b if b.val else a)
                    continue
                work.append((a, b, True))
                work.append((a.bottomRight, b.bottomRight, False))
                work.append((a.bottomLeft, b.bottomLeft, False))
                work.append((a.topRight, b.topRight, False))
                work.append((a.topLeft, b.topLeft, False))
            else:
                br, bl, tr, tl = res.pop(), res.pop(), res.pop(), res.pop()
                if tl.isLeaf and tr.isLeaf and bl.isLeaf and br.isLeaf and tl.val == tr.val == bl.val == br.val:
                    res.append(Node(tl.val, True, None, None, None, None))
                else:
                    res.append(Node(False, False, tl, tr, bl, br))
        return res[0]
```

### Độ phức tạp

- Time: `O(min(N1, N2))` như Cách 2.
- Space: `O(log n)` cho `work`/`res`: độ sâu chỉ `<= 10` và mỗi tầng giữ tối đa vài mục (chưa tính cây kết quả). Lợi thế thật sự của bản lặp là tránh giới hạn độ sâu đệ quy khi cây rất sâu — bài này độ sâu `<= 10` nên không cần, nhưng đáng biết.

## 8. TRACE từng bước

Hàm `trace_intersect(t1, t2)` in từng lần gọi `dfs`: vị trí (đường đi từ gốc, ví dụ `TR.BL`), loại của hai nút, quyết định, và việc nén ở cuối.

Bảng trạng thái cho **ví dụ 1** (đã tính tay ở mục 4):

| Bước | Vị trí | `t1` | `t2` | Quyết định | Kết quả |
|---|---|---|---|---|---|
| 1 | gốc | nút trong | nút trong | đệ quy 4 góc | chờ |
| 2 | TL | lá `1` | lá `1` | `t1` là lá `1` | lá `1` |
| 3 | TR | lá `1` | nút trong | `t1` là lá `1` (bỏ qua cây `t2`) | lá `1` |
| 4 | BL | lá `0` | lá `1` | `t1` là lá `0` nên trả `t2` | lá `1` |
| 5 | BR | lá `0` | lá `0` | `t1` là lá `0` nên trả `t2` | lá `0` |
| 6 | gốc | 4 con: `1,1,1,0` | | không đồng nhất, không nén | nút trong mới |

In [15]:
def describe(nd):
    return f"lá {int(nd.val)}" if nd.isLeaf else "nút trong"


def trace_intersect(t1, t2):
    def dfs(a, b, path, depth):
        pad = "  " * depth
        print(f"{pad}[{path}] t1 = {describe(a)}, t2 = {describe(b)}")
        if a.isLeaf:
            print(f"{pad}  -> t1 là lá {int(a.val)}:", "kết quả = t1 (toàn 1)" if a.val else "kết quả = t2 (0 OR x = x)")
            return a if a.val else b
        if b.isLeaf:
            print(f"{pad}  -> t2 là lá {int(b.val)}:", "kết quả = t2 (toàn 1)" if b.val else "kết quả = t1 (0 OR x = x)")
            return b if b.val else a
        print(f"{pad}  -> cả hai là nút trong, đệ quy 4 góc")
        kids = []
        for name, x, y in zip(["TL", "TR", "BL", "BR"], a.children(), b.children()):
            kids.append(dfs(x, y, path + "." + name if path != "gốc" else name, depth + 1))
        tl, tr, bl, br = kids
        desc = ", ".join(describe(k) for k in kids)
        if all(k.isLeaf for k in kids) and len({k.val for k in kids}) == 1:
            print(f"{pad}  -> 4 con [{desc}] đều là lá cùng giá trị -> NÉN thành lá {int(tl.val)}")
            return leaf(tl.val)
        print(f"{pad}  -> 4 con [{desc}] -> tạo nút trong mới")
        return Node(False, False, tl, tr, bl, br)

    return dfs(t1, t2, "gốc", 0)


res = trace_intersect(t1, t2)
print("kết quả:", serialize(res))

[gốc] t1 = nút trong, t2 = nút trong
  -> cả hai là nút trong, đệ quy 4 góc
  [TL] t1 = lá 1, t2 = lá 1
    -> t1 là lá 1: kết quả = t1 (toàn 1)
  [TR] t1 = lá 1, t2 = nút trong
    -> t1 là lá 1: kết quả = t1 (toàn 1)
  [BL] t1 = lá 0, t2 = lá 1
    -> t1 là lá 0: kết quả = t2 (0 OR x = x)
  [BR] t1 = lá 0, t2 = lá 0
    -> t1 là lá 0: kết quả = t2 (0 OR x = x)
  -> 4 con [lá 1, lá 1, lá 1, lá 0] -> tạo nút trong mới
kết quả: [[0, 0], [1, 1], [1, 1], [1, 1], [1, 0]]


In [16]:
# Một trace có bước nén: hai lưới 4x4 bổ sung nhau (bàn cờ 2x2 và nghịch đảo)
ga = [[1, 0, 1, 0], [0, 1, 0, 1], [1, 0, 1, 0], [0, 1, 0, 1]]
gb = [[0, 1, 0, 1], [1, 0, 1, 0], [0, 1, 0, 1], [1, 0, 1, 0]]
ta, tb = build(ga), build(gb)
print("tree a:", len(serialize(ta)), "phần tử | tree b:", len(serialize(tb)), "phần tử")
res = trace_intersect(ta, tb)
print("kết quả:", serialize(res))

tree a: 21 phần tử | tree b: 21 phần tử
[gốc] t1 = nút trong, t2 = nút trong
  -> cả hai là nút trong, đệ quy 4 góc
  [TL] t1 = nút trong, t2 = nút trong
    -> cả hai là nút trong, đệ quy 4 góc
    [TL.TL] t1 = lá 1, t2 = lá 0
      -> t1 là lá 1: kết quả = t1 (toàn 1)
    [TL.TR] t1 = lá 0, t2 = lá 1
      -> t1 là lá 0: kết quả = t2 (0 OR x = x)
    [TL.BL] t1 = lá 0, t2 = lá 1
      -> t1 là lá 0: kết quả = t2 (0 OR x = x)
    [TL.BR] t1 = lá 1, t2 = lá 0
      -> t1 là lá 1: kết quả = t1 (toàn 1)
    -> 4 con [lá 1, lá 1, lá 1, lá 1] đều là lá cùng giá trị -> NÉN thành lá 1
  [TR] t1 = nút trong, t2 = nút trong
    -> cả hai là nút trong, đệ quy 4 góc
    [TR.TL] t1 = lá 1, t2 = lá 0
      -> t1 là lá 1: kết quả = t1 (toàn 1)
    [TR.TR] t1 = lá 0, t2 = lá 1
      -> t1 là lá 0: kết quả = t2 (0 OR x = x)
    [TR.BL] t1 = lá 0, t2 = lá 1
      -> t1 là lá 0: kết quả = t2 (0 OR x = x)
    [TR.BR] t1 = lá 1, t2 = lá 0
      -> t1 là lá 1: kết quả = t1 (toàn 1)
    -> 4 con [lá 1, lá 

Ở trace thứ hai, mỗi khối `2 x 2` đều được trộn thành `4` lá `1` rồi **nén** thành một lá `1`; đến gốc, 4 con lại là bốn lá `1` nên nén tiếp thành **một lá duy nhất**. Quá trình nén lan lên theo chuỗi — đây là lý do ta kiểm tra nén **ở mọi cấp** (sau mỗi lần đệ quy trả về).

## 9. Code nộp LeetCode

Ta nộp Cách 2 (đệ quy trực tiếp): ngắn gọn, `O(min(N1, N2))`. Lưu ý lớp `Node` ở đây là do notebook định nghĩa ở mục 3.1; trên LeetCode lớp này đã được cho sẵn.

In [17]:
class Solution:
    def intersect(self, quadTree1: 'Node', quadTree2: 'Node') -> 'Node':
        def dfs(t1, t2):
            if t1.isLeaf:
                return t1 if t1.val else t2
            if t2.isLeaf:
                return t2 if t2.val else t1
            tl = dfs(t1.topLeft, t2.topLeft)
            tr = dfs(t1.topRight, t2.topRight)
            bl = dfs(t1.bottomLeft, t2.bottomLeft)
            br = dfs(t1.bottomRight, t2.bottomRight)
            if tl.isLeaf and tr.isLeaf and bl.isLeaf and br.isLeaf and tl.val == tr.val == bl.val == br.val:
                return Node(tl.val, True, None, None, None, None)
            return Node(False, False, tl, tr, bl, br)

        return dfs(quadTree1, quadTree2)


def run_example(q1, q2, expected):
    # mô phỏng cách LeetCode gọi: deserialize hai input, gọi hàm, serialize output
    out = serialize(Solution().intersect(deserialize(q1), deserialize(q2)))
    print("output:", out, "| kỳ vọng:", expected, "| OK" if out == expected else "| SAI")
    return out == expected

assert run_example([[0, 1], [1, 1], [1, 1], [1, 0], [1, 0]], ex2_tree, [[0, 0], [1, 1], [1, 1], [1, 1], [1, 0]])
assert run_example([[1, 0]], [[1, 0]], [[1, 0]])

output: [[0, 0], [1, 1], [1, 1], [1, 1], [1, 0]] | kỳ vọng: [[0, 0], [1, 1], [1, 1], [1, 1], [1, 0]] | OK
output: [[1, 0]] | kỳ vọng: [[1, 0]] | OK


## 10. Vì sao đúng

Gọi `grid(T)` là lưới mà cây `T` biểu diễn, và ta chứng minh bằng **quy nạp theo độ cao** rằng `dfs(a, b)` trả về **cây chuẩn** (canonical: nút trong luôn đại diện khối hỗn hợp) của `grid(a) OR grid(b)`.

**(a) Lá `a` có `val = 1`.** `grid(a)` toàn `1`, nên `grid(a) OR grid(b)` toàn `1`; cây chuẩn của khối toàn `1` là lá `1`, và chính `a` là lá `1`. Trả `a` đúng.

**(b) Lá `a` có `val = 0`.** `grid(a)` toàn `0` nên OR không đổi `grid(b)`. Vì `b` là cây hợp lệ (chuẩn) nên chính `b` là đáp án. Trả `b` đúng.

**(c) `b` là lá:** đối xứng.

**(d) Cả hai là nút trong.** Cả hai khối đều hỗn hợp và chia cùng một cách thành 4 góc. OR được thực hiện **độc lập** trong từng góc phần tư. Theo giả thiết quy nạp, 4 kết quả `tl, tr, bl, br` là các cây chuẩn của 4 góc. Có hai khả năng:
- Cả 4 đều là lá **cùng giá trị**: khối kết quả đồng nhất, cây chuẩn là một lá. Ta nén. Đúng.
- Ngược lại: có một con là nút trong (khối con hỗn hợp, kéo theo khối cha hỗn hợp) hoặc 4 lá không cùng giá trị (hỗn hợp). Khối cha **hỗn hợp** nên phải là nút trong với 4 con này. Đúng.

**(e) Đủ nén một cấp:** vì 4 con đã chuẩn, chỉ có trường hợp "4 lá cùng giá trị" mới vi phạm tính chuẩn. Sau khi nén ta trả về lá, để cấp trên tự xét nén tiếp.

Ta kiểm chứng bằng thí nghiệm trực tiếp trên rất nhiều lưới ngẫu nhiên: `to_grid(dfs(a, b)) == grid(a) OR grid(b)` và cây trả về **bằng đúng** cây `build` của lưới OR.

Để so sánh cây ta cần một hàm so sánh **bỏ qua `val` của nút trong** (vì đề chấp nhận `val` tùy ý ở nút trong), cùng bộ sinh lưới ngẫu nhiên có cấu trúc "khối".

In [18]:
def same_tree(a, b):
    # hai cây bằng nhau nếu cùng hình dạng, lá cùng val; val của nút trong bị bỏ qua
    if a is None or b is None:
        return a is b
    if a.isLeaf != b.isLeaf:
        return False
    if a.isLeaf:
        return bool(a.val) == bool(b.val)
    return all(same_tree(x, y) for x, y in zip(a.children(), b.children()))


def count_nodes(node):
    if node is None:
        return 0
    return 1 + sum(count_nodes(c) for c in node.children())


def is_canonical(node):
    if node.isLeaf:
        return all(c is None for c in node.children())
    kids = node.children()
    if any(c is None for c in kids):
        return False
    if not all(is_canonical(c) for c in kids):
        return False
    return not (all(c.isLeaf for c in kids) and len({bool(c.val) for c in kids}) == 1)


def random_grid(n, rng, p_split=0.65, p_one=0.5):
    # lưới "khối": mỗi vùng hoặc được tô một màu, hoặc chia 4 tiếp -> tạo ra cây không quá nhỏ
    g = [[0] * n for _ in range(n)]

    def rec(r, c, size):
        if size == 1 or rng.random() > p_split:
            v = 1 if rng.random() < p_one else 0
            if v:
                for y in range(r, r + size):
                    for x in range(c, c + size):
                        g[y][x] = 1
            return
        h = size // 2
        rec(r, c, h); rec(r, c + h, h); rec(r + h, c, h); rec(r + h, c + h, h)

    rec(0, 0, n)
    return g


def grid_or(g1, g2):
    return [[a | b for a, b in zip(r1, r2)] for r1, r2 in zip(g1, g2)]


rng = random.Random(558)
cnt = 0
for _ in range(300):
    n = rng.choice([1, 2, 4, 8, 16])
    g1, g2 = random_grid(n, rng), random_grid(n, rng)
    a, b = build(g1), build(g2)
    r = Solution().intersect(a, b)
    assert to_grid(r, n) == grid_or(g1, g2)
    assert same_tree(r, build(grid_or(g1, g2)))
    assert is_canonical(r)
    cnt += 1
print("kiểm chứng quy nạp bằng thí nghiệm:", cnt, "cặp lưới, kết quả đúng và chuẩn")

kiểm chứng quy nạp bằng thí nghiệm: 300 cặp lưới, kết quả đúng và chuẩn


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1 Bung lưới | `O(n^2 log n)` | `O(n^2)` |
| 2 Đệ quy trực tiếp | `O(min(N1, N2))` | `O(log n)` + kết quả |
| 3 Bung ảo + nén | `O(N1 + N2)` | `O(log n)` + cây trung gian |
| 4 Bản lặp | `O(min(N1, N2))` | `O(log n)` + kết quả |

`N1`, `N2` là số nút mỗi cây; tối đa `(4^(x+1) - 1) / 3`, tức khoảng `350 000` nút khi `n = 512`. Ta đếm số lần `dfs` thực sự được gọi để thấy nó không vượt `min(N1, N2)`.

In [19]:
def intersect_count_calls(t1, t2):
    calls = [0]

    def dfs(a, b):
        calls[0] += 1
        if a.isLeaf:
            return a if a.val else b
        if b.isLeaf:
            return b if b.val else a
        tl, tr = dfs(a.topLeft, b.topLeft), dfs(a.topRight, b.topRight)
        bl, br = dfs(a.bottomLeft, b.bottomLeft), dfs(a.bottomRight, b.bottomRight)
        if all(k.isLeaf for k in (tl, tr, bl, br)) and len({k.val for k in (tl, tr, bl, br)}) == 1:
            return leaf(tl.val)
        return Node(False, False, tl, tr, bl, br)

    return dfs(t1, t2), calls[0]


rng = random.Random(11)
print("n | N1 | N2 | số lần gọi dfs")
for n in [4, 16, 64, 256]:
    while True:                       # sinh lại nếu cây quá nhỏ (gốc là lá) để thấy rõ khác biệt
        t_a, t_b = build(random_grid(n, rng, 0.9)), build(random_grid(n, rng, 0.9))
        if count_nodes(t_a) > n and count_nodes(t_b) > n:
            break
    _, calls = intersect_count_calls(t_a, t_b)
    N1, N2 = count_nodes(t_a), count_nodes(t_b)
    assert calls <= min(N1, N2)
    print(n, "|", N1, "|", N2, "|", calls)

n | N1 | N2 | số lần gọi dfs
4 | 17 | 17 | 13
16 | 241 | 261 | 189


64 | 4029 | 3541 | 2637


256 | 35917 | 44621 | 19737


## 12. Bộ test

Các nhóm test: ví dụ của đề (qua deserialize/serialize), edge case (lưới `1x1`, một cây là lá `1`/lá `0`, hai cây giống nhau, nén lan nhiều cấp), kiểm tra cây đầu vào **không bị sửa**, đối chiếu **cả 4 cách** với oracle độc lập (`build` của lưới OR) trên input ngẫu nhiên có seed, và một test cỡ lớn `n = 512`.

In [20]:
# 1) Ví dụ của đề
assert serialize(Solution().intersect(deserialize([[0, 1], [1, 1], [1, 1], [1, 0], [1, 0]]), deserialize(ex2_tree))) == \
    [[0, 0], [1, 1], [1, 1], [1, 1], [1, 0]]
assert serialize(Solution().intersect(deserialize([[1, 0]]), deserialize([[1, 0]]))) == [[1, 0]]

# 2) Edge case: 1x1 (4 tổ hợp)
for x in (0, 1):
    for y in (0, 1):
        r = Solution().intersect(leaf(x), leaf(y))
        assert serialize(r) == [[1, x | y]], (x, y)

# một cây là lá 1 -> kết quả là lá 1; là lá 0 -> kết quả giống cây kia
big = build(random_grid(16, random.Random(1)))
assert serialize(Solution().intersect(leaf(1), big)) == [[1, 1]]
assert serialize(Solution().intersect(big, leaf(1))) == [[1, 1]]
assert same_tree(Solution().intersect(leaf(0), big), big)
assert same_tree(Solution().intersect(big, leaf(0)), big)

# hai cây giống nhau -> kết quả giống chính nó
assert same_tree(Solution().intersect(big, big), big)

# bàn cờ và nghịch đảo -> toàn 1 -> nén lan nhiều cấp thành 1 lá
for n in (2, 4, 8, 16, 32):
    chess = [[(r + c) % 2 for c in range(n)] for r in range(n)]
    anti = [[1 - v for v in row] for row in chess]
    r = Solution().intersect(build(chess), build(anti))
    assert serialize(r) == [[1, 1]], n
# bàn cờ OR chính nó -> giữ nguyên
chess8 = [[(r + c) % 2 for c in range(8)] for r in range(8)]
assert same_tree(Solution().intersect(build(chess8), build(chess8)), build(chess8))
print("ví dụ + edge OK")

ví dụ + edge OK


In [21]:
# 3) Cây đầu vào không bị sửa
rng = random.Random(558)
for _ in range(100):
    n = rng.choice([2, 4, 8, 16])
    a, b = build(random_grid(n, rng)), build(random_grid(n, rng))
    sa, sb = serialize(a), serialize(b)
    Solution().intersect(a, b)
    assert serialize(a) == sa and serialize(b) == sb
print("cây đầu vào không bị sửa")

# 4) Đối chiếu 4 cách + oracle độc lập (build của lưới OR) trên input ngẫu nhiên có seed
rng = random.Random(558)
solvers = [intersect_grid, intersect_dfs, intersect_two_pass, intersect_iter, lambda x, y: Solution().intersect(x, y)]
total = 0
for _ in range(600):
    n = rng.choice([1, 2, 4, 8, 16, 32])
    g1 = random_grid(n, rng, p_split=rng.choice([0.4, 0.7, 0.95]), p_one=rng.choice([0.2, 0.5, 0.8]))
    g2 = random_grid(n, rng, p_split=rng.choice([0.4, 0.7, 0.95]), p_one=rng.choice([0.2, 0.5, 0.8]))
    a, b = build(g1), build(g2)
    expected_tree = build(grid_or(g1, g2))
    for fn in solvers:
        r = fn(a, b)
        assert same_tree(r, expected_tree), fn
        assert to_grid(r, n) == grid_or(g1, g2)
        assert is_canonical(r)
    total += 1
print("đối chiếu", total, "cặp lưới x", len(solvers), "cách: OK")

cây đầu vào không bị sửa


đối chiếu 600 cặp lưới x 5 cách: OK


In [22]:
# 5) Trường hợp hai cây có độ cao khác nhau (một bên rất "phẳng", một bên rất sâu)
rng = random.Random(5)
for _ in range(200):
    n = rng.choice([4, 8, 16])
    g1 = random_grid(n, rng, p_split=0.2)        # ít chia -> cây thấp
    g2 = random_grid(n, rng, p_split=0.95)       # chia nhiều -> cây sâu
    a, b = build(g1), build(g2)
    for x, y in ((a, b), (b, a)):
        assert same_tree(Solution().intersect(x, y), build(grid_or(g1, g2)))
print("độ cao khác nhau OK")

# 6) Test cỡ lớn n = 512 (tối đa theo đề)
rng = random.Random(512)
n = 512
g1, g2 = random_grid(n, rng, 0.97), random_grid(n, rng, 0.97)
a, b = build(g1), build(g2)
t0 = time.perf_counter()
r = Solution().intersect(a, b)
dt = time.perf_counter() - t0
assert to_grid(r, n) == grid_or(g1, g2)
assert same_tree(r, build(grid_or(g1, g2)))
print(f"n=512: N1={count_nodes(a)}, N2={count_nodes(b)}, kết quả {count_nodes(r)} nút, giải trong {dt:.3f}s")

độ cao khác nhau OK


n=512: N1=269789, N2=260281, kết quả 230345 nút, giải trong 0.049s


## 13. Các lỗi người mới hay gặp

Ta tạo vài hàm sai cố ý rồi so với oracle `build(lưới OR)` để thấy lỗi lộ ra như thế nào. Hàm `find_mismatch(fn)` thử nhiều cặp lưới ngẫu nhiên và trả về trường hợp đầu tiên sai.

In [23]:
def find_mismatch(fn, trials=500, seed=99):
    rng = random.Random(seed)
    for _ in range(trials):
        n = rng.choice([2, 4, 8])
        g1, g2 = random_grid(n, rng), random_grid(n, rng)
        a, b = build(g1), build(g2)
        try:
            r = fn(a, b)
            ok = same_tree(r, build(grid_or(g1, g2)))
        except Exception as e:
            ok = False
        if not ok:
            return g1, g2
    return None

### Lỗi 1: quên nén 4 lá cùng giá trị

Chỉ ghép 4 con mà không kiểm tra có đồng nhất không. Kết quả đúng về **ý nghĩa lưới** nhưng sai về **hình dạng cây** (không phải biểu diễn chuẩn), nên không khớp đáp án mong đợi.

In [24]:
def wrong_no_compress(t1, t2):
    if t1.isLeaf:
        return t1 if t1.val else t2
    if t2.isLeaf:
        return t2 if t2.val else t1
    return Node(False, False,
                wrong_no_compress(t1.topLeft, t2.topLeft), wrong_no_compress(t1.topRight, t2.topRight),
                wrong_no_compress(t1.bottomLeft, t2.bottomLeft), wrong_no_compress(t1.bottomRight, t2.bottomRight))

a = build([[1, 0], [0, 1]])
b = build([[0, 1], [1, 0]])
print("sai :", serialize(wrong_no_compress(a, b)))       # gốc nút trong + 4 lá 1
print("đúng:", serialize(Solution().intersect(a, b)))    # một lá 1
print("tìm thấy lưới sai:", find_mismatch(wrong_no_compress) is not None)

sai : [[0, 0], [1, 1], [1, 1], [1, 1], [1, 1]]
đúng: [[1, 1]]
tìm thấy lưới sai: True


**Vì sao sai:** hai lưới `[[1,0],[0,1]]` và `[[0,1],[1,0]]` OR ra toàn `1`; theo định nghĩa cây phải là **một lá** `[1,1]`, nhưng code sai trả nút trong với 4 lá `1`. **Code đúng:** sau khi có `tl, tr, bl, br`, nếu cả 4 là lá cùng `val` thì trả một lá mới.

### Lỗi 2: nhầm khi gặp lá `0`

Viết `if t1.isLeaf: return t1` (hoặc luôn trả lá) mà quên rằng lá `0` **không** quyết định kết quả — kết quả phải là cây còn lại.

In [25]:
def wrong_leaf_zero(t1, t2):
    if t1.isLeaf:
        return t1                        # SAI: lá 0 phải nhường cho t2
    if t2.isLeaf:
        return t2                        # SAI tương tự
    tl = wrong_leaf_zero(t1.topLeft, t2.topLeft)
    tr = wrong_leaf_zero(t1.topRight, t2.topRight)
    bl = wrong_leaf_zero(t1.bottomLeft, t2.bottomLeft)
    br = wrong_leaf_zero(t1.bottomRight, t2.bottomRight)
    if tl.isLeaf and tr.isLeaf and bl.isLeaf and br.isLeaf and tl.val == tr.val == bl.val == br.val:
        return leaf(tl.val)
    return Node(False, False, tl, tr, bl, br)

g1, g2 = [[0, 0], [0, 0]], [[1, 0], [0, 1]]
print("sai :", serialize(wrong_leaf_zero(build(g1), build(g2))))
print("đúng:", serialize(Solution().intersect(build(g1), build(g2))))

sai : [[1, 0]]
đúng: [[0, 0], [1, 1], [1, 0], [1, 0], [1, 1]]


**Vì sao sai:** `0 OR x = x`, mà code sai trả luôn lá `0`, nên mất thông tin của `x`. **Code đúng:** `return t1 if t1.val else t2`.

### Lỗi 3: dùng `val` của nút **trong** như thể có nghĩa

Đề nói rõ: với nút trong, `val` là **giá trị tùy ý**. Ví dụ ngay trong ví dụ 1 của đề, nút gốc của `quadTree1` là `[0,1]` (`isLeaf = 0`, `val = 1`). Nếu viết `if t1.val: return t1` mà không kiểm tra `isLeaf` thì sẽ trả nguyên cả cây `t1` vì `val = 1` ở nút trong.

In [26]:
def wrong_use_internal_val(t1, t2):
    if t1.val:                           # SAI: chưa kiểm tra isLeaf, val của nút trong là vô nghĩa
        return t1
    if t2.val:
        return t2
    if t1.isLeaf:
        return t2
    if t2.isLeaf:
        return t1
    tl = wrong_use_internal_val(t1.topLeft, t2.topLeft)
    tr = wrong_use_internal_val(t1.topRight, t2.topRight)
    bl = wrong_use_internal_val(t1.bottomLeft, t2.bottomLeft)
    br = wrong_use_internal_val(t1.bottomRight, t2.bottomRight)
    if tl.isLeaf and tr.isLeaf and bl.isLeaf and br.isLeaf and tl.val == tr.val == bl.val == br.val:
        return leaf(tl.val)
    return Node(False, False, tl, tr, bl, br)

q1 = deserialize([[0, 1], [1, 1], [1, 1], [1, 0], [1, 0]])     # gốc [0,1]: nút trong có val = 1
print("sai :", serialize(wrong_use_internal_val(q1, deserialize(ex2_tree))))
print("đúng:", serialize(Solution().intersect(q1, deserialize(ex2_tree))))

sai : [[0, 1], [1, 1], [1, 1], [1, 0], [1, 0]]
đúng: [[0, 0], [1, 1], [1, 1], [1, 1], [1, 0]]


**Vì sao sai:** ở ví dụ 1 code sai trả lại **nguyên cây `quadTree1`**. **Code đúng:** luôn kiểm tra `isLeaf` **trước** khi đọc `val`, đúng như `if t1.isLeaf: return t1 if t1.val else t2`.

### Lỗi 4: kiểm tra "cùng giá trị" nhưng quên kiểm tra các con có phải lá

Nén chỉ hợp lệ khi **cả 4 con là lá**. Nếu chỉ so `val`, các nút trong (có `val` tùy ý, ở code của ta là `False`) bị nhầm là cùng giá trị.

In [27]:
def wrong_only_vals(t1, t2):
    if t1.isLeaf:
        return t1 if t1.val else t2
    if t2.isLeaf:
        return t2 if t2.val else t1
    tl = wrong_only_vals(t1.topLeft, t2.topLeft)
    tr = wrong_only_vals(t1.topRight, t2.topRight)
    bl = wrong_only_vals(t1.bottomLeft, t2.bottomLeft)
    br = wrong_only_vals(t1.bottomRight, t2.bottomRight)
    if tl.val == tr.val == bl.val == br.val:     # SAI: thiếu kiểm tra isLeaf của 4 con
        return leaf(tl.val)
    return Node(False, False, tl, tr, bl, br)

# tìm phản ví dụ nhỏ nhất trong các lưới 4x4
rng = random.Random(99)
cands = []
for _ in range(400):
    g1, g2 = random_grid(4, rng), random_grid(4, rng)
    if not same_tree(wrong_only_vals(build(g1), build(g2)), build(grid_or(g1, g2))):
        cands.append((len(serialize(build(grid_or(g1, g2)))), g1, g2))
cands.sort(key=lambda x: x[0])
_, g1, g2 = cands[0]
print("số phản ví dụ trong 400 cặp lưới 4x4:", len(cands))
print("g1 =", g1)
print("g2 =", g2)
print("sai :", serialize(wrong_only_vals(build(g1), build(g2))))
print("đúng:", serialize(Solution().intersect(build(g1), build(g2))))

số phản ví dụ trong 400 cặp lưới 4x4: 17
g1 = [[0, 0, 1, 0], [0, 0, 1, 1], [0, 1, 0, 0], [0, 1, 0, 0]]
g2 = [[0, 0, 1, 0], [1, 1, 1, 1], [1, 1, 0, 0], [0, 0, 0, 0]]
sai : [[1, 0]]
đúng: [[0, 0], [0, 0], [0, 0], [0, 0], [1, 0], [1, 0], [1, 0], [1, 1], [1, 1], [1, 1], [1, 0], [1, 1], [1, 1], [1, 1], [1, 1], [1, 0], [1, 1]]


**Vì sao sai:** khi 4 con đều là **nút trong** (cùng `val = False`) hoặc lẫn lộn, code sai nén chúng thành một lá `0` và làm mất toàn bộ phần cấu trúc bên dưới (hoặc làm sai lưới). **Code đúng:** điều kiện nén là `tl.isLeaf and tr.isLeaf and bl.isLeaf and br.isLeaf and tl.val == tr.val == bl.val == br.val`.

## 14. Mẹo & nhận dạng pattern

- Bài "kết hợp hai cây cùng cấu trúc" (merge, OR, AND, XOR, so sánh, cộng) → duyệt **song song** hai cây bằng đệ quy `dfs(a, b)`; chia trường hợp theo "nút nào là lá".
- Tính chất **hấp thụ** giúp cắt tỉa mạnh: `1 OR x = 1`, `0 OR x = x` (và `0 AND x = 0`, `1 AND x = x`). Hãy luôn tự hỏi "gặp nút lá thì kết quả có suy ra ngay được không?".
- Cây **chuẩn** (canonical) đòi hỏi **nén ngược** sau khi đệ quy (cùng bước với bài dựng quad-tree LeetCode 427).
- Đừng bao giờ tin vào `val` của nút trong; chỉ đọc `val` khi `isLeaf` là `True`.
- Luôn truyền đủ 6 đối số khi tạo `Node` để chạy được với mọi phiên bản định nghĩa lớp.
- Khi test, so sánh cây bằng hàm **bỏ qua `val` nút trong**, và đối chiếu với `build` của lưới trong oracle.
- Nếu bài nhắc "serialize kiểu LeetCode": hãy nhớ lá sinh ra 4 `null` ở tầng sau, và các `null` thừa ở cuối được bỏ.

## 15. Biến thể

### 15.1. AND của hai quad-tree

Đối ngẫu: lá `0` thống trị (`0 AND x = 0`), lá `1` nhường (`1 AND x = x`). Còn lại giữ nguyên, kể cả bước nén.

In [28]:
def quad_and(t1, t2):
    if t1.isLeaf:
        return t2 if t1.val else t1
    if t2.isLeaf:
        return t1 if t2.val else t2
    tl, tr = quad_and(t1.topLeft, t2.topLeft), quad_and(t1.topRight, t2.topRight)
    bl, br = quad_and(t1.bottomLeft, t2.bottomLeft), quad_and(t1.bottomRight, t2.bottomRight)
    if all(k.isLeaf for k in (tl, tr, bl, br)) and len({k.val for k in (tl, tr, bl, br)}) == 1:
        return leaf(tl.val)
    return Node(False, False, tl, tr, bl, br)

rng = random.Random(15)
for _ in range(200):
    n = rng.choice([2, 4, 8, 16])
    g1, g2 = random_grid(n, rng), random_grid(n, rng)
    g_and = [[x & y for x, y in zip(r1, r2)] for r1, r2 in zip(g1, g2)]
    assert same_tree(quad_and(build(g1), build(g2)), build(g_and))
print("AND OK")

AND OK


### 15.2. Phủ định (NOT / đảo bit)

Đảo mọi ô: lá đổi `val`, nút trong đảo cả 4 con. Cây chuẩn vẫn chuẩn sau khi đảo (khối hỗn hợp vẫn hỗn hợp), nên không cần nén.

In [29]:
def invert(node):
    if node.isLeaf:
        return leaf(not node.val)
    return Node(False, False, *[invert(c) for c in node.children()])

rng = random.Random(152)
for _ in range(100):
    n = rng.choice([1, 2, 4, 8])
    g = random_grid(n, rng)
    inv = [[1 - v for v in row] for row in g]
    assert same_tree(invert(build(g)), build(inv))
print("NOT OK")

NOT OK


### 15.3. Đếm số ô `1` ngay trên cây (không bung lưới)

Một lá `1` có diện tích `size * size`, ta cộng thẳng. Thời gian `O(N)` theo số nút thay vì `O(n^2)`.

In [30]:
def count_ones(node, size):
    if node.isLeaf:
        return size * size if node.val else 0
    h = size // 2
    return sum(count_ones(c, h) for c in node.children())

rng = random.Random(153)
for _ in range(100):
    n = rng.choice([1, 2, 4, 8, 16])
    g = random_grid(n, rng)
    assert count_ones(build(g), n) == sum(map(sum, g))
print("count_ones OK")

count_ones OK


### 15.4. Các bài liên quan

- LeetCode 427 *Construct Quad Tree*: dựng quad-tree từ lưới (đúng hàm `build` ở mục 3.2).
- LeetCode 1008, 105, 106: dựng cây từ dãy duyệt (cùng tinh thần "dựng cấu trúc từ biểu diễn").
- LeetCode 617 *Merge Two Binary Trees*: duyệt song song hai cây nhị phân — mẫu rất gần với bài này.
- LeetCode 100 *Same Tree*: so sánh hai cây song song.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 Bung lưới | OR từng ô rồi `build` lại | `O(n^2 log n)` | `O(n^2)` | oracle kiểm thử; hiểu đề |
| 2 Đệ quy trực tiếp | lá `1` thống trị, lá `0` nhường, nén khi 4 lá giống nhau | `O(min(N1, N2))` | `O(log n)` | **cách nộp** |
| 3 Bung ảo + nén | coi lá như 4 con giống nó; nén ở pass 2 | `O(N1 + N2)` | `O(log n)` + cây trung gian | khi muốn tách trộn khỏi nén cho dễ đọc |
| 4 Bản lặp | hai ngăn xếp (việc cần làm, kết quả) | `O(min(N1, N2))` | `O(log n)` | khi cần tránh đệ quy (cây rất sâu) |

## 17. Bài tập vận dụng

Làm lần lượt từ dễ đến khó. Gợi ý ở mục 18, đáp án ở mục 19.

### Bài 1 (đoán output)

Với `build` và `serialize` ở mục 3, đoán kết quả `serialize(Solution().intersect(build(g1), build(g2)))` cho 3 cặp lưới `2 x 2` sau, rồi điền vào `ex1_answer` (danh sách 3 danh sách serialize). **Không** chạy trước khi đoán.

1. `g1 = [[0,0],[0,0]]`, `g2 = [[0,0],[0,0]]`
2. `g1 = [[1,0],[0,0]]`, `g2 = [[0,0],[0,1]]`
3. `g1 = [[1,0],[0,1]]`, `g2 = [[0,1],[1,0]]`

In [31]:
ex1_pairs = [
    ([[0, 0], [0, 0]], [[0, 0], [0, 0]]),
    ([[1, 0], [0, 0]], [[0, 0], [0, 1]]),
    ([[1, 0], [0, 1]], [[0, 1], [1, 0]]),
]
ex1_answer = None     # ví dụ [[[1, 0]], [[1, 1]], [[1, 1]]]  <- chỉ là mẫu định dạng, không phải đáp án

### Bài 2 (tính tay)

Cho hai lưới `4 x 4` dưới đây. Tính tay lưới OR, rồi cho biết cây quad-tree chuẩn của lưới OR có **bao nhiêu nút** và **bao nhiêu lá**. Điền `ex2_answer = (số_nút, số_lá)`.

In [32]:
ex2_g1 = [[1, 0, 0, 0],
          [0, 0, 0, 0],
          [0, 0, 1, 1],
          [0, 0, 1, 1]]
ex2_g2 = [[0, 1, 0, 0],
          [0, 0, 0, 0],
          [0, 0, 0, 0],
          [1, 0, 0, 0]]
ex2_answer = None     # ví dụ (0, 0)

### Bài 3 (điền TODO)

Hoàn thành `ex3_intersect`. Có hai chỗ `TODO`: điều kiện khi `t1` là lá và điều kiện nén. Cell này **không gọi** hàm.

In [33]:
def ex3_intersect(t1, t2):
    if t1.isLeaf:
        # TODO 1: trả về gì nếu t1 là lá? (phụ thuộc t1.val)
        pass
    if t2.isLeaf:
        return t2 if t2.val else t1
    tl = ex3_intersect(t1.topLeft, t2.topLeft)
    tr = ex3_intersect(t1.topRight, t2.topRight)
    bl = ex3_intersect(t1.bottomLeft, t2.bottomLeft)
    br = ex3_intersect(t1.bottomRight, t2.bottomRight)
    # TODO 2: điều kiện để nén 4 con thành một lá
    if False:
        return Node(tl.val, True, None, None, None, None)
    return Node(False, False, tl, tr, bl, br)

### Bài 4 (tự viết lại)

Tự viết **không nhìn** hàm `ex4_rewrite(t1, t2)` (đệ quy trực tiếp). Để `pass` nếu chưa làm.

In [34]:
def ex4_rewrite(t1, t2):
    pass

### Bài 5 (biến thể / mini challenge)

Viết `ex5_xor(t1, t2)`: quad-tree của **XOR** hai lưới. Gợi ý nhanh: lá `0` nhường (`0 XOR x = x`), lá `1` thì kết quả là **đảo** cây còn lại (`1 XOR x = NOT x`), hai lá thì `a ^ b`; vẫn phải nén. (Hàm `invert` ở mục 15.2 dùng được.)

In [35]:
def ex5_xor(t1, t2):
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: chuyển mỗi lưới sang cây rồi nghĩ trên **lưới OR** trước.
- Mức 2: nếu lưới OR đồng nhất thì kết quả là một lá `[[1, v]]`; còn không thì `[[0,0]]` rồi 4 con (mỗi con lá `[1, v]`).
- Mức 3: (1) toàn `0` nên `[[1, 0]]`; (2) OR là `[[1,0],[0,1]]`, hỗn hợp nên `[[0,0],[1,1],[1,0],[1,0],[1,1]]` (gốc rồi 4 lá TL=1, TR=0, BL=0, BR=1); (3) OR toàn `1` nên `[[1, 1]]`.

**Bài 2**
- Mức 1: OR từng ô rồi chia lưới `4 x 4` thành bốn khối `2 x 2`.
- Mức 2: khối đồng nhất là một lá (1 nút); khối hỗn hợp là nút trong cộng 4 lá (5 nút).
- Mức 3: lưới OR có hai khối hỗn hợp, một khối toàn `0`, một khối toàn `1`: nút = `1 + 5 + 1 + 5 + 1`.

**Bài 3**
- Mức 1: lá `1` thống trị, lá `0` nhường.
- Mức 2: TODO 1 là `return t1 if t1.val else t2`.
- Mức 3: TODO 2 là `tl.isLeaf and tr.isLeaf and bl.isLeaf and br.isLeaf and tl.val == tr.val == bl.val == br.val`.

**Bài 4**
- Mức 1: viết ra 3 trường hợp bằng lời (`t1` lá, `t2` lá, cả hai nút trong).
- Mức 2: nhớ bước nén sau khi đệ quy và đọc `val` chỉ khi `isLeaf`.
- Mức 3: so sánh với `intersect_grid` ở mục 5 trên nhiều lưới ngẫu nhiên.

**Bài 5**
- Mức 1: làm giống OR nhưng đổi quy tắc ở lá.
- Mức 2: `t1` lá `0` trả `t2`; `t1` lá `1` trả `invert(t2)`; làm đối xứng cho `t2`.
- Mức 3: hai lá thì trả `leaf(bool(t1.val) != bool(t2.val))`; sau khi đệ quy vẫn nén như OR.

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [36]:
ex1_solution = [[[1, 0]], [[0, 0], [1, 1], [1, 0], [1, 0], [1, 1]], [[1, 1]]]
got = [serialize(Solution().intersect(build(g1), build(g2))) for g1, g2 in ex1_pairs]
assert got == ex1_solution, got
print("Bài 1 OK:", ex1_solution)

Bài 1 OK: [[[1, 0]], [[0, 0], [1, 1], [1, 0], [1, 0], [1, 1]], [[1, 1]]]


### Đáp án Bài 2

In [37]:
or2 = grid_or(ex2_g1, ex2_g2)
print("lưới OR:"); [print("  ", row) for row in or2]
res2 = Solution().intersect(build(ex2_g1), build(ex2_g2))

def count_leaves(nd):
    return 1 if nd.isLeaf else sum(count_leaves(c) for c in nd.children())

ex2_solution = (13, 10)
assert (count_nodes(res2), count_leaves(res2)) == ex2_solution
print("Bài 2 OK:", ex2_solution)

lưới OR:
   [1, 1, 0, 0]
   [0, 0, 0, 0]
   [0, 0, 1, 1]
   [1, 0, 1, 1]
Bài 2 OK: (13, 10)


### Đáp án Bài 3

In [38]:
def ex3_solution(t1, t2):
    if t1.isLeaf:
        return t1 if t1.val else t2
    if t2.isLeaf:
        return t2 if t2.val else t1
    tl = ex3_solution(t1.topLeft, t2.topLeft)
    tr = ex3_solution(t1.topRight, t2.topRight)
    bl = ex3_solution(t1.bottomLeft, t2.bottomLeft)
    br = ex3_solution(t1.bottomRight, t2.bottomRight)
    if tl.isLeaf and tr.isLeaf and bl.isLeaf and br.isLeaf and tl.val == tr.val == bl.val == br.val:
        return Node(tl.val, True, None, None, None, None)
    return Node(False, False, tl, tr, bl, br)

rng = random.Random(3)
for _ in range(200):
    n = rng.choice([1, 2, 4, 8, 16])
    g1, g2 = random_grid(n, rng), random_grid(n, rng)
    assert same_tree(ex3_solution(build(g1), build(g2)), build(grid_or(g1, g2)))
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [39]:
def ex4_solution(t1, t2):
    if t1.isLeaf:
        return t1 if t1.val else t2
    if t2.isLeaf:
        return t2 if t2.val else t1
    kids = [ex4_solution(x, y) for x, y in zip(t1.children(), t2.children())]
    if all(k.isLeaf for k in kids) and len({bool(k.val) for k in kids}) == 1:
        return leaf(kids[0].val)
    return Node(False, False, *kids)

rng = random.Random(4)
for _ in range(200):
    n = rng.choice([1, 2, 4, 8, 16])
    g1, g2 = random_grid(n, rng), random_grid(n, rng)
    assert same_tree(ex4_solution(build(g1), build(g2)), intersect_grid(build(g1), build(g2)))
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [40]:
def ex5_solution(t1, t2):
    if t1.isLeaf and t2.isLeaf:
        return leaf(bool(t1.val) != bool(t2.val))
    if t1.isLeaf:
        return invert(t2) if t1.val else t2
    if t2.isLeaf:
        return invert(t1) if t2.val else t1
    kids = [ex5_solution(x, y) for x, y in zip(t1.children(), t2.children())]
    if all(k.isLeaf for k in kids) and len({bool(k.val) for k in kids}) == 1:
        return leaf(kids[0].val)
    return Node(False, False, *kids)

rng = random.Random(5)
for _ in range(300):
    n = rng.choice([1, 2, 4, 8, 16])
    g1, g2 = random_grid(n, rng), random_grid(n, rng)
    g_xor = [[x ^ y for x, y in zip(r1, r2)] for r1, r2 in zip(g1, g2)]
    r = ex5_solution(build(g1), build(g2))
    assert same_tree(r, build(g_xor)) and is_canonical(r)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def intersect(self, quadTree1: 'Node', quadTree2: 'Node') -> 'Node':
        def dfs(t1, t2):
            if t1.isLeaf:                              # lá 1: toàn 1; lá 0: nhường cây kia
                return t1 if t1.val else t2
            if t2.isLeaf:
                return t2 if t2.val else t1
            tl = dfs(t1.topLeft, t2.topLeft)           # cả hai là nút trong: đệ quy 4 góc
            tr = dfs(t1.topRight, t2.topRight)
            bl = dfs(t1.bottomLeft, t2.bottomLeft)
            br = dfs(t1.bottomRight, t2.bottomRight)
            if tl.isLeaf and tr.isLeaf and bl.isLeaf and br.isLeaf and tl.val == tr.val == bl.val == br.val:
                return Node(tl.val, True, None, None, None, None)     # nén
            return Node(False, False, tl, tr, bl, br)
        return dfs(quadTree1, quadTree2)
```

Câu thần chú: **"lá 1 thắng, lá 0 nhường; cả hai nút trong thì đệ quy 4 góc; bốn lá cùng màu thì nén."**

### Checklist trước khi submit

- [ ] Đọc `val` **chỉ khi** `isLeaf` là `True`.
- [ ] Lá `1` trả lá đó; lá `0` trả cây còn lại (cả hai phía `t1` và `t2`).
- [ ] Đệ quy đúng 4 góc, đúng vị trí tương ứng (`topLeft` với `topLeft`, ...).
- [ ] Nén khi 4 con đều là lá **và** cùng `val`.
- [ ] Tạo `Node` với đủ 6 đối số (`Node(val, isLeaf, tl, tr, bl, br)`), nút trong để `val = False`.
- [ ] Test: lưới `1x1`, một cây là lá `1`, một cây là lá `0`, hai bàn cờ bổ sung nhau (nén lan lên gốc), cây đầu vào không bị sửa.
- [ ] Skeleton đúng: `class Solution`, method `intersect(self, quadTree1: 'Node', quadTree2: 'Node') -> 'Node'`.